# Setup

In [1]:
from getpass import getpass
GITHUB_TOKEN = getpass("")

··········


In [2]:
import pandas as pd
import numpy as np
import requests
import io
import time
import json
import math

GITHUB_RAW = "https://raw.githubusercontent.com/W2NJL/nwr-gap-research/main"
headers = {"Authorization": f"token {GITHUB_TOKEN}"}

def fetch_csv(path):
    r = requests.get(f"{GITHUB_RAW}/{path}", headers=headers)
    r.raise_for_status()
    return pd.read_csv(io.StringIO(r.text))

# --- NWR station data ---
wx = fetch_csv("wx_stations.csv")
wx = wx[wx['country'] == 'USA'].dropna(subset=['latitude', 'longitude']).copy()
wx['longitude'] = -wx['longitude'].abs()
print(f"{len(wx)} US NWR stations loaded.")

# --- Hazard event datasets ---
coastal    = fetch_csv("data/coastal_flood_events.csv")
wildfires  = fetch_csv("data/wildfire_events.csv")
hurricanes = fetch_csv("data/us_hurricane_landfalls.csv")
tornadoes  = fetch_csv("data/tornadoes_f3plus.csv")
print(f"Hazard events: {len(coastal)} coastal flood, {len(wildfires)} wildfire, "
      f"{len(hurricanes)} hurricane landfalls, {len(tornadoes)} tornadoes F3+")

1029 US NWR stations loaded.
Hazard events: 4591 coastal flood, 8440 wildfire, 607 hurricane landfalls, 3278 tornadoes F3+


In [3]:
# --- US cities (population centers)
cities = pd.read_csv("https://raw.githubusercontent.com/CSC-2053-Spring-26-100/lab-12-finding-the-weather-radio-gaps/main/us_cities.csv")
cities = cities[~cities['state_id'].isin(['AK', 'HI', 'PR'])].reset_index(drop=True)
print(f"{len(cities)} contiguous US cities loaded.")

# --- NRI (county-level multi-hazard risk)
nri = pd.read_csv("/content/NRI_Table_Counties.csv")
print(f"{len(nri)} NRI county rows loaded.")

# --- SVI (tract-level social vulnerability)
svi = pd.read_csv("/content/SVI_2022_US.csv")
print(f"{len(svi)} SVI tract rows loaded.")

# --- RUCC (county rurality)
RUCC_URL = "https://ers.usda.gov/sites/default/files/_laserfiche/DataFiles/53251/Ruralurbancontinuumcodes2023.xlsx?v=59358"
import urllib.request
urllib.request.urlretrieve(RUCC_URL, "rucc2023.xlsx")
rucc = pd.read_excel("rucc2023.xlsx")
rucc.columns = rucc.columns.str.strip()
print(f"{len(rucc)} RUCC county rows loaded.")

4834 contiguous US cities loaded.
3232 NRI county rows loaded.
84120 SVI tract rows loaded.
3235 RUCC county rows loaded.


In [4]:
RADIOLAND_BASE = "http://52.151.197.43/search_stream"
COVERAGE_THRESHOLD = 50.0

def query_nwr_coverage(lat, lon, rx_height=10, min_sig_strength=3, verbose=False):
    """
    Query the RadioLand API for NWR stations receivable at (lat, lon).
    Returns a DataFrame sorted by field_strength descending, or None on failure.
    """
    params = {
        "lat": lat, "lon": lon,
        "search_freq": "none", "callsign": "none", "request_type": 1,
        "pi_code": "none", "sig_strength": min_sig_strength, "am_sig_strength": 2,
        "startMiles": "none", "miles": "null", "slogan": "none", "owner": "none",
        "format": "none", "wfo": "none", "rxHeight": rx_height, "mlbTeam": "none",
        "market": "none", "country": "none", "sp": "none", "measurementUnit": "metric",
        "locationName": "", "broadcastBand": "WX", "timeOfDay": "day", "model": "longley_rice",
    }
    try:
        resp = requests.get(RADIOLAND_BASE, params=params, stream=True, timeout=90)
        resp.raise_for_status()
    except requests.RequestException as e:
        if verbose:
            print(f"Request failed: {e}")
        return None

    for raw_line in resp.iter_lines():
        if not raw_line:
            continue
        line = raw_line.decode('utf-8')
        if not line.startswith('data: '):
            continue
        event = json.loads(line[6:])
        if event.get('type') == 'progress' and verbose:
            print(f"  [{event['percentage']:>3}%] {event['message']}", end='\r')
        elif event.get('type') == 'complete':
            if verbose:
                print("  [100%] Done.                                        ")
            result = json.loads(event['data'])
            df = pd.DataFrame(result['data'])
            if df.empty:
                return df
            for col in ['lon', 'transmitter_lon']:
                if col in df.columns:
                    df[col] = -df[col].abs()
            return df.sort_values('field_strength', ascending=False).reset_index(drop=True)
    return None


def get_best_signal(lat, lon, threshold=COVERAGE_THRESHOLD):
    df = query_nwr_coverage(lat, lon)
    if df is None or df.empty:
        return {'best_callsign': None, 'best_field_strength': 0.0,
                'best_distance_km': None, 'stations_above_threshold': 0, 'covered': False}
    best = df.iloc[0]
    return {
        'best_callsign':            best['callsign'],
        'best_field_strength':      float(best['field_strength']),
        'best_distance_km':         float(best.get('distance', float('nan'))),
        'stations_above_threshold': int((df['field_strength'] >= threshold).sum()),
        'covered':                  float(best['field_strength']) >= threshold,
    }

In [5]:
test = get_best_signal(39.8732, -74.6643)
print(test)

{'best_callsign': 'KIH-28', 'best_field_strength': 64.59, 'best_distance_km': 26.86, 'stations_above_threshold': 1, 'covered': True}


# Full City Sweep

In [6]:
import os

OUTFILE = "full_city_sweep.csv"
START_FROM = 0

# Resume support: if a partial file exists, pick up where it left off
if os.path.exists(OUTFILE):
    done_df = pd.read_csv(OUTFILE)
    START_FROM = len(done_df)
    results = done_df.to_dict('records')
    print(f"Resuming from row {START_FROM} ({len(results)} already done)")
else:
    results = []

for i in range(START_FROM, len(cities)):
    row = cities.iloc[i]
    print(f"[{i+1}/{len(cities)}] {row['city']}, {row['state_id']}...", end=' ', flush=True)
    r = get_best_signal(row['lat'], row['lng'])
    r.update({
        'city': row['city'], 'state_id': row['state_id'],
        'lat': row['lat'], 'lng': row['lng'],
        'population': row['population'], 'county_name': row['county_name'],
    })
    results.append(r)
    print("COVERED" if r['covered'] else "GAP", f"({r['best_field_strength']} dB)")

    if (i + 1) % 100 == 0:
        pd.DataFrame(results).to_csv(OUTFILE, index=False)
        print(f"--- checkpoint saved at row {i+1} ---")

    time.sleep(1)

pd.DataFrame(results).to_csv(OUTFILE, index=False)
print(f"\nDone. Saved {len(results)} rows to {OUTFILE}")

Resuming from row 4834 (4834 already done)

Done. Saved 4834 rows to full_city_sweep.csv


# Stage 2: Build rural grid with already included cities filtered out.

In [7]:
GRID_SPACING = 0.5
EXCLUSION_RADIUS_MI = 30

lat_vals = np.arange(25.0, 49.5, GRID_SPACING)
lon_vals = np.arange(-125.0, -66.5, GRID_SPACING)
grid_lat, grid_lon = np.meshgrid(lat_vals, lon_vals)
grid_points = pd.DataFrame({'lat': grid_lat.ravel(), 'lon': grid_lon.ravel()})
print(f"Raw grid: {len(grid_points)} points")

def min_dist_to_cities_miles(glat, glon, city_lats, city_lons):
    R = 3958.8
    p1 = np.radians(glat)
    p2 = np.radians(city_lats)
    dphi = np.radians(city_lats - glat)
    dlambda = np.radians(city_lons - glon)
    a = np.sin(dphi/2)**2 + np.cos(p1)*np.cos(p2)*np.sin(dlambda/2)**2
    d = 2*R*np.arcsin(np.sqrt(a))
    return d.min()

city_lats = cities['lat'].values
city_lngs = cities['lng'].values

grid_points['min_dist_mi'] = grid_points.apply(
    lambda r: min_dist_to_cities_miles(r['lat'], r['lon'], city_lats, city_lngs), axis=1
)

rural_grid = grid_points[grid_points['min_dist_mi'] > EXCLUSION_RADIUS_MI].reset_index(drop=True)
print(f"Filtered rural grid: {len(rural_grid)} points (removed {len(grid_points)-len(rural_grid)} near-city points)")

est_hours = len(rural_grid) * 4.4 / 3600
print(f"Estimated time: {est_hours:.1f} hours")

Raw grid: 5733 points
Filtered rural grid: 3376 points (removed 2357 near-city points)
Estimated time: 4.1 hours


In [8]:
GRID_OUTFILE = "rural_grid_sweep.csv"
START_FROM = 0

if os.path.exists(GRID_OUTFILE):
    done_df = pd.read_csv(GRID_OUTFILE)
    START_FROM = len(done_df)
    grid_results = done_df.to_dict('records')
    print(f"Resuming from row {START_FROM} ({len(grid_results)} already done)")
else:
    grid_results = []

for i in range(START_FROM, len(rural_grid)):
    row = rural_grid.iloc[i]
    print(f"[{i+1}/{len(rural_grid)}] ({row['lat']:.2f}, {row['lon']:.2f})...", end=' ', flush=True)
    r = get_best_signal(row['lat'], row['lon'])
    r.update({'lat': row['lat'], 'lon': row['lon'], 'min_dist_to_city_mi': row['min_dist_mi']})
    grid_results.append(r)
    print("COVERED" if r['covered'] else "GAP", f"({r['best_field_strength']} dB)")

    if (i + 1) % 100 == 0:
        pd.DataFrame(grid_results).to_csv(GRID_OUTFILE, index=False)
        print(f"--- checkpoint saved at row {i+1} ---")

    time.sleep(1)

pd.DataFrame(grid_results).to_csv(GRID_OUTFILE, index=False)
print(f"\nDone. Saved {len(grid_results)} rows to {GRID_OUTFILE}")

Resuming from row 3376 (3376 already done)

Done. Saved 3376 rows to rural_grid_sweep.csv


# Joins and Scores

In [25]:

STATE_ABBR = {
    'Alabama': 'AL', 'Arizona': 'AZ', 'Arkansas': 'AR', 'California': 'CA',
    'Colorado': 'CO', 'Connecticut': 'CT', 'Delaware': 'DE', 'Florida': 'FL',
    'Georgia': 'GA', 'Idaho': 'ID', 'Illinois': 'IL', 'Indiana': 'IN',
    'Iowa': 'IA', 'Kansas': 'KS', 'Kentucky': 'KY', 'Louisiana': 'LA',
    'Maine': 'ME', 'Maryland': 'MD', 'Massachusetts': 'MA', 'Michigan': 'MI',
    'Minnesota': 'MN', 'Mississippi': 'MS', 'Missouri': 'MO', 'Montana': 'MT',
    'Nebraska': 'NE', 'Nevada': 'NV', 'New Hampshire': 'NH', 'New Jersey': 'NJ',
    'New Mexico': 'NM', 'New York': 'NY', 'North Carolina': 'NC', 'North Dakota': 'ND',
    'Ohio': 'OH', 'Oklahoma': 'OK', 'Oregon': 'OR', 'Pennsylvania': 'PA',
    'Rhode Island': 'RI', 'South Carolina': 'SC', 'South Dakota': 'SD', 'Tennessee': 'TN',
    'Texas': 'TX', 'Utah': 'UT', 'Vermont': 'VT', 'Virginia': 'VA',
    'Washington': 'WA', 'West Virginia': 'WV', 'Wisconsin': 'WI', 'Wyoming': 'WY',
    'District of Columbia': 'DC',
}

SVI_ACCESS_VARS = ['EP_NOVEH', 'EP_LIMENG', 'EP_AGE65', 'EP_DISABL', 'EP_UNINSUR']
EVENT_RADIUS_MI = 25
RURALITY_WEIGHT = 1.25  # extra weight given to rurality vs hazard/access


def clean_county_name(s):
    """Normalize county/parish/independent-city naming across RUCC, NRI, SVI, and us_cities.csv."""
    s = str(s)
    s = s.replace(' County', '')
    s = s.replace(' Parish', '')   # Louisiana
    s = s.replace(' (city)', '')   # us_cities.csv VA format
    s = s.replace(' city', '')     # RUCC's VA format (lowercase, no parens)
    s = s.replace(' City', '')     # capitalized variant
    return s.strip()


def haversine_vec(lat1, lon1, lats2, lons2):
    """Vectorized haversine distance (miles) from one point to an array of points."""
    R = 3958.8
    p1 = np.radians(lat1)
    p2 = np.radians(lats2)
    dphi = np.radians(lats2 - lat1)
    dlambda = np.radians(lons2 - lon1)
    a = np.sin(dphi / 2) ** 2 + np.cos(p1) * np.cos(p2) * np.sin(dlambda / 2) ** 2
    return 2 * R * np.arcsin(np.sqrt(a))


def count_nearby_events(lat, lon, event_lats, event_lons, radius_mi=EVENT_RADIUS_MI):
    return int((haversine_vec(lat, lon, event_lats, event_lons) <= radius_mi).sum())


# --- 3a: Rurality multiplier (RUCC) ---
rucc_clean = rucc.copy()
rucc_clean['County_Name'] = rucc_clean['County_Name'].apply(clean_county_name)
rucc_clean['rurality_multiplier'] = 1.0 + (rucc_clean['RUCC_2023'] - 1) / 8 * 1.0

# --- 3b: Hazard multiplier, NRI component (RISK_SCORE) ---
nri_clean = nri.copy()
nri_clean['state_id'] = nri_clean['STATE'].map(STATE_ABBR)
nri_clean['COUNTY'] = nri_clean['COUNTY'].apply(clean_county_name)
nri_clean['hazard_multiplier'] = 1.0 + (nri_clean['RISK_SCORE'] / 100) * 1.0

# --- 3c: Access multiplier (SVI, county-aggregated, -999 sentinel filtered) ---
svi_clean = svi.copy()
for col in SVI_ACCESS_VARS:
    svi_clean[col] = svi_clean[col].replace(-999, np.nan)

svi_county = svi_clean.groupby(['ST_ABBR', 'STCNTY'], as_index=False)[SVI_ACCESS_VARS].mean()
county_names = svi_clean.groupby('STCNTY')['COUNTY'].first().reset_index()
svi_county = svi_county.merge(county_names, on='STCNTY', how='left')
svi_county['COUNTY'] = svi_county['COUNTY'].apply(clean_county_name)
svi_county['access_index'] = svi_county[SVI_ACCESS_VARS].mean(axis=1)
svi_county['access_multiplier'] = 1.0 + (svi_county['access_index'] / 100) * 1.0

# --- 3d: Join all three multipliers onto the city sweep ---
city_scored = full_city_sweep.copy()
city_scored['county_name_clean'] = city_scored['county_name'].apply(clean_county_name)

city_scored = city_scored.merge(
    rucc_clean[['State', 'County_Name', 'rurality_multiplier']].drop_duplicates(subset=['State', 'County_Name']),
    left_on=['state_id', 'county_name_clean'], right_on=['State', 'County_Name'], how='left'
).drop(columns=['State', 'County_Name'])

city_scored = city_scored.merge(
    nri_clean[['state_id', 'COUNTY', 'hazard_multiplier']].drop_duplicates(subset=['state_id', 'COUNTY']),
    left_on=['state_id', 'county_name_clean'], right_on=['state_id', 'COUNTY'], how='left'
).drop(columns=['COUNTY'])

city_scored = city_scored.merge(
    svi_county[['ST_ABBR', 'COUNTY', 'access_multiplier']].drop_duplicates(subset=['ST_ABBR', 'COUNTY']),
    left_on=['state_id', 'county_name_clean'], right_on=['ST_ABBR', 'COUNTY'], how='left'
).drop(columns=['ST_ABBR', 'COUNTY'])

matched_mask = city_scored[['rurality_multiplier', 'hazard_multiplier', 'access_multiplier']].notna().all(axis=1)
city_scored_final = city_scored[matched_mask].copy()
print(f"Stage 3d: {matched_mask.sum()} / {len(city_scored)} cities fully matched "
      f"({len(city_scored) - matched_mask.sum()} dropped, mostly CT planning-region mismatches)")

# --- 3f/3g: Historical hazard event proximity, folded into hazard_multiplier ---
event_sources = {
    'coastal_flood_count': (coastal['lat'].values, coastal['lon'].values),
    'wildfire_count':      (wildfires['lat'].values, wildfires['lon'].values),
    'hurricane_count':     (hurricanes['lat'].values, hurricanes['lon'].values),
    'tornado_count':       (tornadoes['slat'].values, tornadoes['slon'].values),
}

for col, (ev_lats, ev_lons) in event_sources.items():
    city_scored_final[col] = city_scored_final.apply(
        lambda r: count_nearby_events(r['lat'], r['lng'], ev_lats, ev_lons), axis=1
    )

pct_cols = []
for col in event_sources:
    pct_col = f'{col}_pct'
    city_scored_final[pct_col] = city_scored_final[col].rank(pct=True)
    pct_cols.append(pct_col)

city_scored_final['event_exposure_score'] = city_scored_final[pct_cols].mean(axis=1)

# NRI stays the primary hazard signal (70%); recent historical event density adds a smaller (30%) bonus
city_scored_final['hazard_multiplier'] = (
    1.0
    + 0.7 * (city_scored_final['hazard_multiplier'] - 1.0)
    + 0.3 * city_scored_final['event_exposure_score']
)

print(f"Stage 3g: hazard_multiplier now blends NRI RISK_SCORE (70%) with "
      f"25-mile historical event density across all 4 hazard datasets (30%)")



Stage 3d: 4812 / 4834 cities fully matched (22 dropped, mostly CT planning-region mismatches)
Stage 3g: hazard_multiplier now blends NRI RISK_SCORE (70%) with 25-mile historical event density across all 4 hazard datasets (30%)


# Priority Score

In [26]:
COVERAGE_THRESHOLD = 50.0  # dB, must match the threshold used in Stage 1/2 sweeps

city_scored_final['gap_depth'] = (COVERAGE_THRESHOLD - city_scored_final['best_field_strength']).clip(lower=0)

city_scored_final['combined_multiplier'] = (
    RURALITY_WEIGHT * city_scored_final['rurality_multiplier']
    + city_scored_final['hazard_multiplier']
    + city_scored_final['access_multiplier']
) / (RURALITY_WEIGHT + 2)

city_scored_final['priority_score'] = (
    city_scored_final['population']
    * city_scored_final['gap_depth']
    * city_scored_final['combined_multiplier']
)

gap_cities = city_scored_final[city_scored_final['covered'] == False].copy()
print(f"Stage 4: {len(gap_cities)} gap cities scored, "
      f"priority_score range [{gap_cities['priority_score'].min():,.0f}, {gap_cities['priority_score'].max():,.0f}]")


Stage 4: 480 gap cities scored, priority_score range [806, 21,838,449]


# Top 50

In [27]:

top50 = gap_cities.sort_values('priority_score', ascending=False).head(50).reset_index(drop=True)
top50.index += 1  # 1-indexed rank

display_cols = ['city', 'state_id', 'county_name', 'population', 'best_field_strength',
                 'gap_depth', 'rurality_multiplier', 'hazard_multiplier', 'access_multiplier',
                 'combined_multiplier', 'priority_score']
print(top50[display_cols].to_string())

top50.to_csv('top50_priority_cities.csv', index=True)
city_scored_final.to_csv('city_scored_final_full.csv', index=False)
print("\nSaved top50_priority_cities.csv and city_scored_final_full.csv")

                city state_id          county_name  population  best_field_strength  gap_depth  rurality_multiplier  hazard_multiplier  access_multiplier  combined_multiplier  priority_score
1        Bakersfield       CA                 Kern    567585.0                21.18      28.82                1.125           1.837064           1.095592             1.335048    2.183845e+07
2              Provo       UT                 Utah    501690.0                33.73      16.27                1.125           1.765424           1.059266             1.301828    1.062616e+07
3           Murrieta       CA            Riverside    476630.0                34.37      15.63                1.000           1.797485           1.098507             1.275690    9.503542e+06
4              Ogden       UT                Weber    571457.0                37.22      12.78                1.125           1.707313           1.079928             1.290305    9.423381e+06
5         Scottsdale       AZ             Mar